# Clase 10 — Seguimiento de experimentos con MLflow

Entrenar un modelo produce un resultado. Rastrear el experimento conserva la evidencia necesaria para entender **qué se ejecutó, con qué configuración, sobre qué datos y qué resultado produjo**.

En esta clase compararás `LinearRegression` y `RandomForestRegressor` sobre el mismo problema de duración de viajes de NYC Green Taxi. Primero representarás el registro de forma manual y después crearás dos runs reproducibles con MLflow.

Al terminar podrás:

- distinguir experimento, run, parámetro, métrica, tag, dataset y artifact;
- reconocer las fortalezas y límites del seguimiento manual y automático;
- registrar dos modelos comparables en un servidor local de MLflow;
- interpretar la comparación sin confundir Tracking con Model Registry.


# 1. Motivación

Imagina que estás desarrollando la receta perfecta de galletas con chispas de chocolate. En un intento aumentas la harina; en otro agregas más chispas; después pruebas con nueces. Tras una docena de recetas, recordar sólo “la última quedó bien” no permite saber qué cambio produjo el resultado.

Con modelos de machine learning ocurre lo mismo. Dos ejecuciones pueden cambiar:

- el algoritmo o sus hiperparámetros;
- los datos de entrenamiento o validación;
- el código de preparación;
- las versiones de las dependencias;
- la métrica o el criterio usado para elegir.

Una tabla puede ser el primer registro. La siguiente representa un historial simulado de seis runs del mismo experimento:

| Run | Modelo | Parámetros clave | Entrenamiento → validación | RMSE de validación | Tiempo de entrenamiento | Tamaño del artifact | Decisión |
|---|---|---|---|---:|---:|---:|---|
| `run-001` | Mean baseline | sin hiperparámetros | Mar → Apr 2026 | 8.42 min | 0.01 s | 0.001 MiB | referencia mínima |
| `run-002` | Linear Regression | `fit_intercept=True` | Mar → Apr 2026 | 6.18 min | 0.05 s | 0.005 MiB | conservar como baseline entrenado |
| `run-003` | Decision Tree | `max_depth=10`, `min_samples_leaf=8` | Mar → Apr 2026 | 5.71 min | 0.08 s | 0.31 MiB | revisar sobreajuste |
| `run-004` | Random Forest | `n_estimators=100`, `max_depth=12`, `min_samples_leaf=5` | Mar → Apr 2026 | 5.12 min | 0.18 s | 1.58 MiB | candidato equilibrado |
| `run-005` | Gradient Boosting | `n_estimators=200`, `learning_rate=0.05`, `max_depth=4` | Mar → Apr 2026 | 4.98 min | 1.24 s | 2.36 MiB | mejor RMSE; medir inferencia |
| `run-006` | Neural Network | `layers=[128,64]`, `dropout=0.2`, `epochs=50` | Mar → Apr 2026 | 5.04 min | 28.40 s | 4.70 MiB | costo no justificado |

Todas las filas conservan entrenamiento en marzo, validación en abril, las mismas cinco features, el mismo target y RMSE. Así podemos atribuir las diferencias al modelo y sus parámetros. La última columna registra la interpretación: el menor RMSE no siempre produce la mejor decisión.

![Experiment tracking within the machine learning lifecycle](../assets/modulo-02-ciclo-mlops/clase-10/mlops-experiment-tracking.png)

*Experiment tracking conecta las decisiones de desarrollo con la evidencia producida por cada ejecución.*


# 2. Definiciones

## 2.1 Conceptos principales

- **Seguimiento de experimentos:** proceso de registrar y organizar sistemáticamente los insumos, la configuración y las salidas de los entrenamientos de ML.
- **Experimento:** grupo lógico de runs que responden una pregunta común. En esta clase, `nyc-taxi-clase-10` reúne las ejecuciones que comparan modelos de duración.
- **Run:** una ejecución identificable de entrenamiento y evaluación. Cada vez que ejecutamos el script se crean runs nuevos.
- **Parámetro:** entrada o decisión de configuración, por ejemplo `max_depth=12`.
- **Métrica:** medición producida por la ejecución, por ejemplo `validation_rmse=5.10`.
- **Tag:** etiqueta descriptiva que facilita búsqueda y organización, por ejemplo `task=trip-duration-regression`.
- **Dataset:** referencia a los datos usados y a su función, como `training` o `validation`.
- **Artifact:** archivo producido por un run: un pipeline entrenado, una gráfica, predicciones o un reporte.
- **Metadatos:** información que describe el run, incluidos parámetros, métricas, tags, timestamps y versiones.

## 2.2 Pregunta, hipótesis, insumos y salidas

La pregunta del experimento es:

> Con los mismos datos, features y métrica, ¿Random Forest reduce el RMSE de validación lo suficiente para justificar un entrenamiento más lento y un artifact más grande que Linear Regression?

La hipótesis comprobable es:

> Random Forest reducirá el RMSE por lo menos 0.5 minutos, pero tardará más en entrenar y producirá un artifact de al menos diez veces el tamaño de Linear Regression.

La hipótesis anticipa resultados observables; no afirma de antemano qué modelo debe elegirse. El diseño separa lo que permanece constante de lo que cambia:

| Papel | Elementos del experimento |
|---|---|
| Constantes | marzo para entrenamiento, abril para validación, cinco features, target y RMSE |
| Cambio | estimador e hiperparámetros |
| Salidas | RMSE, tiempo de entrenamiento, tamaño estimado y pipeline entrenado |

El seguimiento ayuda a la reproducibilidad, pero no reemplaza el control de versiones del código, la identificación de los datos ni un ambiente reproducible.


# 3. ¿Lo necesito?

Un rastreador de experimentos ayuda a responder:

- **visión general:** ¿qué ejecuciones realizamos?;
- **detalle:** ¿qué datos, código y configuración produjo cada resultado?;
- **comparación:** ¿qué cambio mejoró o empeoró el modelo?;
- **revisión:** ¿otra persona puede entender y repetir el recorrido?;
- **diagnóstico:** si el rendimiento cambia, ¿podemos investigar por qué?

Para uno o dos resultados exploratorios puede bastar una nota estructurada. El valor de una herramienta aumenta cuando crecen los runs, los artifacts, el equipo o la necesidad de buscar y comparar evidencia.

Tracking no corrige una evaluación mal diseñada ni convierte datos deficientes en datos confiables. Registra lo que le indiquemos; por eso importa decidir bien qué conservar.


# 4. ¿Cómo rastrear experimentos de machine learning?

El seguimiento puede hacerse manualmente o automatizarse. Las tres aproximaciones más comunes son:

1. hojas de cálculo, notas y convenciones de nombres;
2. código propio que agrega una fila y guarda archivos en cada ejecución;
3. herramientas especializadas para tracking.

## 4.1 Seguimiento manual

Un equipo puede guardar métricas y parámetros en una hoja de cálculo y usar nombres como `random_forest_depth12_rmse510.pkl` para relacionar resultados con archivos.

Este enfoque permite comenzar con poca infraestructura, pero tiene límites:

- requiere disciplina y tiempo en cada ejecución;
- copiar resultados a mano introduce errores;
- una nota puede perderse o quedar desincronizada del artifact;
- dos personas pueden sobrescribir información;
- las convenciones de nombres se vuelven largas e inconsistentes;
- buscar y comparar cientos de runs resulta difícil.

![Manual experiment log in a spreadsheet](../assets/modulo-02-ciclo-mlops/clase-10/experiment-tracking-spreadsheet.svg)

*La hoja reproduce los seis registros de la sección 1. Las columnas de RMSE, tiempo, tamaño y decisión permiten comparar cada run sin reducir la elección a una sola métrica.*

## 4.2 Seguimiento automático sin una herramienta especializada

El mismo registro puede automatizarse desde Python. El programa crea un diccionario con los insumos y salidas del run, lo agrega a un `DataFrame` y guarda el resultado:

```python
from pathlib import Path

import pandas as pd

ruta_registro = Path("experimentos.csv")
if ruta_registro.exists():
    registro = pd.read_csv(ruta_registro)
else:
    registro = pd.DataFrame()

run = {
    "model": "random_forest",
    "max_depth": 12,
    "validation_rmse": 5.10,
}

registro = pd.concat([registro, pd.DataFrame([run])], ignore_index=True)
registro.to_csv(ruta_registro, index=False)
```

Esto reduce errores de captura, pero todavía debemos diseñar IDs, concurrencia, almacenamiento de artifacts, búsqueda y relación entre archivos y filas.

Git complementa este enfoque al versionar código y archivos pequeños. No sustituye un tracker: un commit no representa automáticamente un run, y no conviene guardar bases de tracking ni modelos grandes en el repositorio.

## 4.3 Herramientas especializadas

Herramientas como [MLflow](https://mlflow.org/), [Weights & Biases](https://wandb.ai/), [Comet](https://www.comet.com/) y [TensorBoard](https://www.tensorflow.org/tensorboard) ofrecen mecanismos para registrar, buscar, visualizar y comparar experimentos.

![Comparison of experiment tracking tools](../assets/modulo-02-ciclo-mlops/clase-10/tracking-tools-historical-neptune.png)

*La captura histórica permite reconocer criterios como colaboración, integración, artifacts y visualización; no representa la disponibilidad ni los precios actuales.*

Neptune aparece porque la captura se creó cuando todavía era un producto disponible. OpenAI [anunció su adquisición en diciembre de 2025](https://openai.com/index/openai-to-acquire-neptune/) para incorporar su experiencia en observación y comparación de entrenamientos a la infraestructura interna de OpenAI. Después de la adquisición, Neptune dejó de operar como servicio SaaS independiente: su aplicación alojada y su API [cerraron el 5 de marzo de 2026](https://docs.neptune.ai/transition_hub), y el sitio quedó como centro de transición con guías de exportación y migración. Por esa razón, Neptune ya no se considera una opción vigente para adoptar en el curso, aunque la comparación histórica sigue siendo útil.

En esta clase usamos MLflow porque permite observar localmente el recorrido completo entre código, runs, métricas y artifacts.


# 5. Mejores prácticas para el seguimiento de experimentos

## 5.1 Qué conviene rastrear siempre

1. **Hipótesis o pregunta:** qué decisión intenta informar el run. Sin una pregunta, acumular ejecuciones no produce aprendizaje.
2. **Código:** scripts de preparación, entrenamiento y evaluación, junto con su versión o commit cuando corresponda.
3. **Entorno:** Python y dependencias reproducibles. En este repositorio quedan declaradas en `pyproject.toml` y bloqueadas en `uv.lock`.
4. **Datos:** fuente, periodo, filtros, número de filas, features, target y una referencia que permita identificar la versión.
5. **Parámetros:** configuración del algoritmo y decisiones de preparación que cambien entre runs.
6. **Métricas:** nombre, conjunto sobre el que se calculan y unidad. `validation_rmse=5.10 minutos` es más claro que `error=5.10`.
7. **Artifacts:** pipeline entrenado, firma, ejemplo de entrada, gráficas y otros archivos necesarios para interpretar el resultado.
8. **Decisión:** qué se aprendió y cuál es el siguiente paso. El tracker conserva evidencia; la conclusión todavía requiere criterio.

## 5.2 Reglas para comparaciones confiables

- Define la pregunta antes de ejecutar.
- Cambia una decisión a la vez cuando quieras atribuir su efecto.
- Conserva los mismos datos, features, target y métrica entre candidatos.
- Separa entrenamiento y validación; no selecciones un modelo con la métrica de entrenamiento.
- Usa nombres descriptivos como `linear_regression` y `random_forest`.
- No sobrescribas runs: una nueva ejecución crea nueva evidencia.
- Registra también candidatos descartados; sin ellos se pierde la historia de la decisión.
- No guardes secretos, tokens, datos personales ni rutas privadas en tags o artifacts.

## 5.3 Información adicional según el problema

Para datos estructurados puede ser útil registrar:

- una muestra o resumen del esquema de entrada;
- distribuciones de features y predicciones;
- residuos y casos con mayor error;
- importancia de features o explicaciones;
- tiempo de inferencia y consumo de recursos;
- resultados de varias semillas cuando el algoritmo sea estocástico.

En deep learning también suelen registrarse checkpoints, curvas por época, uso de GPU y ejemplos de mejores o peores predicciones. En NLP pueden añadirse tokenizer, prompts y métricas específicas. No se trata de guardar todo indiscriminadamente, sino la evidencia necesaria para responder la pregunta del experimento.


# 6. MLflow

![MLflow logo](../assets/modulo-02-ciclo-mlops/clase-10/mlflow-logo.png)

## 6.1 ¿Qué es MLflow?

[MLflow](https://mlflow.org/docs/latest/ml/) es una plataforma open source para construir, evaluar, versionar y llevar modelos de machine learning hacia producción. No reemplaza a scikit-learn, PyTorch o TensorFlow: se integra con esas bibliotecas para conservar y conectar la evidencia que producen.

Para modelos de ML, la documentación oficial organiza la plataforma en cinco capacidades principales:

| Capacidad | Qué aporta |
|---|---|
| **Tracking & Experiments** | registra y compara runs, parámetros, métricas y artifacts |
| **Model Registry** | organiza versiones, aliases, linaje y ciclo de vida de modelos seleccionados |
| **Model Deployment** | sirve modelos mediante APIs y otros destinos de inferencia |
| **ML Library Integrations** | conecta MLflow con bibliotecas como scikit-learn, PyTorch y TensorFlow |
| **Model Evaluation** | calcula, conserva y compara evidencia de evaluación |

El componente relevante para registrar y comparar las ejecuciones de este caso es **Tracking & Experiments**.

## 6.2 Cómo organiza MLflow los experimentos

[MLflow Tracking](https://mlflow.org/docs/latest/ml/tracking/) es una **API y una interfaz web** para registrar parámetros, versiones de código, métricas y archivos producidos al ejecutar código de machine learning, y para consultar después esa evidencia.

Su unidad central es el **run**: una ejecución de código con identidad, tiempo de inicio y fin, parámetros, métricas, tags y artifacts. Varios runs que responden una misma pregunta se agrupan en un **experiment**.

En nuestro caso:

- el experiment se llama `nyc-taxi-clase-10`;
- cada ejecución de un candidato produce un run;
- marzo de 2026 se registra como dataset de entrenamiento y abril como dataset de validación;
- parámetros, RMSE, tiempo, tamaño y pipeline quedan asociados al mismo run;
- la UI permite abrir y comparar esa evidencia sin reconstruirla a partir de archivos sueltos.

## 6.3 ¿Qué ocurre cuando registramos un run?

El script importa el **MLflow Python SDK**, es decir, la biblioteca cliente. Sus funciones forman la **Tracking API**: operaciones como iniciar un run o registrar una métrica.

En la configuración local, esas operaciones viajan por HTTP al **Tracking Server**. El servidor distribuye la información según su naturaleza:

```text
registrar_experimentos.py
└── MLflow Python SDK / Tracking API
    └── HTTP → Tracking Server (127.0.0.1:5000)
        ├── metadatos → mlflow.db
        ├── archivos  → mlartifacts/
        └── consultas → Tracking UI en el navegador
```

- `mlflow.db` es el **Backend Store**: conserva IDs, tiempos, parámetros, métricas y tags.
- `mlartifacts/` es el **Artifact Store**: conserva modelos y otros archivos asociados a los runs.
- La **Tracking UI** es la vista web que el mismo servidor construye con esa información.

![MLflow Tracking components and user interface](../assets/modulo-02-ciclo-mlops/clase-10/mlflow-tracking-basics.png)

La imagen representa la misma separación: el código registra evidencia, el servidor la persiste y la interfaz permite consultarla.

## 6.4 Operaciones de Tracking que usa el script

El script usa logging explícito para hacer visible qué información se registra. Las llamadas pertenecen a la **Tracking API de Python**:

| Operación | Qué hace en nuestra práctica |
|---|---|
| `mlflow.set_tracking_uri(...)` | configura la dirección del Tracking Server al que se conectará el SDK |
| `mlflow.set_experiment(...)` | selecciona el experiment o lo crea si todavía no existe |
| `mlflow.start_run(...)` | abre el contexto de un run; al salir del bloque registra su finalización |
| `mlflow.log_params(...)` | guarda configuración e hiperparámetros como pares clave–valor |
| `mlflow.log_metrics(...)` | guarda resultados numéricos que podrán ordenarse y compararse |
| `mlflow.set_tags(...)` | agrega contexto descriptivo para filtrar o interpretar el run |
| `mlflow.log_input(...)` | vincula metadatos de un dataset con su función de entrenamiento o validación |
| `mlflow.sklearn.log_model(...)` | empaqueta y guarda el pipeline entrenado como un artifact de MLflow Models |

`mlflow.sklearn.log_model(...)` conserva el pipeline dentro del run como un artifact; no crea por sí solo una entrada en Model Registry.

La Tracking UI coloca lado a lado la evidencia de varios runs:

![MLflow user interface comparing multiple runs](../assets/modulo-02-ciclo-mlops/clase-10/mlflow-run-comparison.png)

La distribución visual puede cambiar entre versiones; los conceptos experiment, run, parameters, metrics y artifacts permanecen.


## 6.5 Datos y modelos del caso

Los CSV proceden de los Parquet oficiales de [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page): marzo de 2026 aporta 1,200 filas de entrenamiento y abril aporta 500 de validación. La selección es reproducible con `random_state=42`.

La siguiente celda entrena ambos pipelines sin iniciar MLflow. El resultado sólo vive en la salida del notebook: todavía no existe un historial consultable.


In [ ]:
from pathlib import Path
import sys
from time import perf_counter

import pandas as pd
from sklearn.metrics import root_mean_squared_error

ROOT = Path.cwd()
if not (ROOT / "labs").exists():
    ROOT = ROOT.parent
STARTER = ROOT / "labs/starters/clase-10-experiment-tracking"
sys.path.insert(0, str(STARTER))

from modelos import construir_pipeline
from preparar_datos import FEATURES, TARGET, cargar_muestra

train = cargar_muestra(STARTER / "datos" / "green-taxi-train.csv")
valid = cargar_muestra(STARTER / "datos" / "green-taxi-validation.csv")

resultados = []
for nombre in ("linear_regression", "random_forest"):
    pipeline = construir_pipeline(nombre)
    inicio = perf_counter()
    pipeline.fit(train[FEATURES], train[TARGET])
    tiempo = perf_counter() - inicio
    predicciones = pipeline.predict(valid[FEATURES])
    resultados.append(
        {
            "model": nombre,
            "validation_rmse": root_mean_squared_error(
                valid[TARGET], predicciones
            ),
            "training_time_seconds": tiempo,
        }
    )

pd.DataFrame(resultados).round(3)


# 7. Práctica: registrar dos runs con MLflow

La práctica ocurre en `labs/trabajo-local/clase-10/`. Esa ruta está ignorada por Git y permite crear la base, artifacts y scripts sin modificar el starter.

## 7.1 Preparar la carpeta de trabajo

Ejecuta desde la raíz del repositorio:

```bash
mkdir -p labs/trabajo-local/clase-10
cp -R labs/starters/clase-10-experiment-tracking/. labs/trabajo-local/clase-10/
```

- `mkdir` crea un directorio.
- `-p` crea también los directorios intermedios que falten y evita un error si la ruta ya existe.
- `cp` copia archivos.
- `-R` copia recursivamente el contenido de los subdirectorios, incluido `datos/`.
- El `/.` al final del origen significa “copia todo el contenido”, incluidos archivos ocultos, sin crear otra carpeta anidada con el nombre del starter.

Comprueba la copia:

```bash
ls labs/trabajo-local/clase-10
```

`ls` muestra el contenido de la ruta indicada. Debes ver `README.md`, `datos/`, `modelos.py` y `preparar_datos.py`.

## 7.2 Sincronizar el ambiente

```bash
uv sync --locked
```

- `uv sync` hace que `.venv` coincida con las dependencias declaradas por el proyecto.
- `--locked` exige usar exactamente `uv.lock`; si el lockfile no coincide, el comando falla en lugar de modificarlo.

No ejecutes `uv add`: MLflow y scikit-learn ya están declarados por el profesor.


## 7.3 Iniciar el Tracking Server

En una terminal, desde la raíz del repositorio, ejecuta:

```bash
uv run mlflow server \
  --backend-store-uri sqlite:///labs/trabajo-local/clase-10/mlflow.db \
  --artifacts-destination ./labs/trabajo-local/clase-10/mlartifacts \
  --host 127.0.0.1 \
  --port 5000
```

Este comando tiene varias partes nuevas:

| Parte | Función |
|---|---|
| `uv run` | ejecuta el programa dentro del ambiente administrado por el proyecto |
| `mlflow server` | inicia el servidor de Tracking y la interfaz web |
| `--backend-store-uri` | define dónde se guardan experiments, runs, parámetros, métricas y tags |
| `sqlite:///.../mlflow.db` | selecciona SQLite y una ruta relativa a la raíz del repositorio |
| `--artifacts-destination` | define el directorio donde el servidor guardará modelos y otros archivos |
| `--host 127.0.0.1` | acepta conexiones únicamente desde esta computadora |
| `--port 5000` | publica el servicio local en el puerto 5000 |
| `\` | continúa el mismo comando en la línea siguiente en Git Bash, Zsh y Bash |

Esta configuración coincide con el patrón local vigente de MLflow: SQLite funciona como Backend Store para los metadatos y `--artifacts-destination` dirige los artifacts a un directorio local servido por el Tracking Server. La separación está documentada en [MLflow Tracking Server](https://mlflow.org/docs/latest/self-hosting/architecture/tracking-server/).

### Si aparece un error relacionado con SQLite

MLflow usa el módulo `sqlite3` de Python; no necesita que el comando `sqlite3` esté disponible en la terminal. Verifica primero el módulo que realmente utilizará:

```bash
uv run python -c "import sqlite3; print(sqlite3.sqlite_version)"
```

La opción `-c` pide a Python ejecutar la instrucción escrita entre comillas. Si imprime una versión, SQLite ya está disponible y no debes instalar nada.

Si aparece `ModuleNotFoundError: No module named 'sqlite3'`, instala SQLite según tu sistema y vuelve a ejecutar la verificación:

| Sistema | Instalación |
|---|---|
| Windows | Descarga **Precompiled Binaries for Windows** desde [SQLite Download Page](https://www.sqlite.org/download.html) y extrae `sqlite3.exe`. Si el módulo de Python sigue ausente, reinstala Python 3.12 desde [python.org](https://www.python.org/downloads/). |
| macOS | Ejecuta `brew install sqlite`. Homebrew lo instala separado del SQLite incluido por macOS. |
| Ubuntu o Debian | Ejecuta `sudo apt update` y después `sudo apt install sqlite3 libsqlite3-dev`. |
| Fedora | Ejecuta `sudo dnf install sqlite sqlite-devel`. |

El comando `sqlite3 --version` comprueba la herramienta de terminal; la prueba con `uv run python -c ...` confirma que el intérprete usado por MLflow puede abrir la base. Si sólo funciona la primera, debe corregirse la instalación de Python 3.12.

Mantén esa terminal abierta. El proceso ocupa la sesión mientras el servidor funciona.

Abre `http://127.0.0.1:5000` en el navegador. Si la página no carga, revisa que la terminal muestre el servidor activo y que ningún otro proceso esté usando el puerto 5000.


## 7.4 Actividad en clase: completar el tracking con la documentación

Completa esta actividad consultando únicamente la documentación oficial de
MLflow. No uses ChatGPT, Copilot, Claude ni otro agente de IA para generar o
completar el código.

Fuentes permitidas:

- [MLflow Tracking Quickstart](https://mlflow.org/docs/latest/ml/tracking/quickstart/)
- [MLflow Tracking APIs](https://mlflow.org/docs/latest/ml/tracking/tracking-api/)
- [Python API: `mlflow`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.html)
- [Python API: `mlflow.sklearn`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.sklearn.html)

Usa el buscador de esas páginas para localizar cada función, revisar sus
argumentos y adaptar los ejemplos al experimento de NYC Taxi.

En otra terminal, entra a la carpeta local:

```bash
cd labs/trabajo-local/clase-10
```

`cd` cambia el directorio actual. Desde esta ubicación, crea
`registrar_experimentos.py` en VS Code y completa el siguiente esqueleto con tu
equipo. Los módulos del starter resuelven la preparación y la construcción de
modelos; este archivo se concentra en medir y registrar cada run.

Los `...` son placeholders. Sustitúyelos durante la construcción del script; el
archivo no está listo para ejecutarse mientras quede alguno.

```python
from pathlib import Path
from pickle import dumps
from time import perf_counter

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from modelos import construir_pipeline
from preparar_datos import FEATURES, TARGET, cargar_muestra
from sklearn.metrics import root_mean_squared_error

TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENTO = "nyc-taxi-clase-10"
MODELOS = ("linear_regression", "random_forest")


def registrar_modelo(
    nombre_modelo,
    datos_entrenamiento,
    datos_validacion,
):
    pipeline = construir_pipeline(nombre_modelo)
    x_train = datos_entrenamiento[FEATURES]
    y_train = datos_entrenamiento[TARGET]
    x_valid = datos_validacion[FEATURES]
    y_valid = datos_validacion[TARGET]

    inicio = perf_counter()
    pipeline.fit(x_train, y_train)
    tiempo_entrenamiento = perf_counter() - inicio
    predicciones = pipeline.predict(x_valid)
    rmse = root_mean_squared_error(y_valid, predicciones)
    tamano_pickle_mib = len(dumps(pipeline)) / (1024**2)

    ejemplo_entrada = x_valid.head(5)
    # MLflow infiere el esquema de entrada y salida del modelo.
    # En este caso registra:
    # - las cinco features, con sus nombres y tipos;
    # - el tipo y la forma de las predicciones.
    firma = infer_signature(
        ejemplo_entrada,
        pipeline.predict(ejemplo_entrada),
    )

    with mlflow.start_run(run_name=nombre_modelo):
        parametros = {
            "model_name": nombre_modelo,
            "train_month": "2026-03",
            "validation_month": "2026-04",
            "feature_count": len(FEATURES),
        }
        if nombre_modelo == "random_forest":
            parametros.update(
                {
                    "n_estimators": 100,
                    "max_depth": 12,
                    "min_samples_leaf": 5,
                    "random_state": 42,
                }
            )

        # Convertimos los DataFrames en referencias de datasets de MLflow.
        # Esto no modifica los datos usados por el pipeline.
        dataset_entrenamiento = mlflow.data.from_pandas(
            datos_entrenamiento,
            source=str(ruta_entrenamiento),
            targets=TARGET,
            name="green-taxi-2026-03",
        )

        dataset_validacion = mlflow.data.from_pandas(
            datos_validacion,
            source=str(ruta_validacion),
            targets=TARGET,
            name="green-taxi-2026-04",
        )

        # Vinculamos cada dataset con la función que cumplió dentro del run.
        mlflow.log_input(dataset_entrenamiento, context="training")
        mlflow.log_input(dataset_validacion, context="validation")

        # TODO 1: registrar todos los pares del diccionario `parametros`.
        ...

        metricas = {
            "validation_rmse": rmse,
            "training_time_seconds": tiempo_entrenamiento,
            "estimated_pickle_size_mib": tamano_pickle_mib,
        }
        # TODO 2: registrar todos los pares del diccionario `metricas`.
        ...

        tags = {
            "dataset": "NYC Green Taxi",
            "task": "trip-duration-regression",
            "question": "rf_vs_linear_tradeoff",
            "hypothesis": "rf_reduces_rmse_at_least_0.5_min",
        }
        # TODO 3: registrar todos los pares del diccionario `tags`.
        ...

        # TODO 4: guardar `pipeline` con el nombre "model", `firma`
        # y `ejemplo_entrada` mediante la integración sklearn de MLflow.
        ...

    print(
        f"{nombre_modelo}: RMSE={rmse:.3f}, "
        f"tiempo={tiempo_entrenamiento:.3f}s, "
        f"pickle estimado={tamano_pickle_mib:.3f} MiB"
    )


def main():
    base = Path(__file__).parent
    ruta_train = base / "datos" / "green-taxi-train.csv"
    ruta_valid = base / "datos" / "green-taxi-validation.csv"
    train = cargar_muestra(ruta_train)
    valid = cargar_muestra(ruta_valid)

    mlflow.set_tracking_uri(TRACKING_URI)
    mlflow.set_experiment(EXPERIMENTO)

    for nombre_modelo in MODELOS:
        registrar_modelo(nombre_modelo, train, valid)


if __name__ == "__main__":
    main()
```

Los cuatro placeholders corresponden a operaciones de MLflow. Antes de
ejecutar, confirma que no quede ningún `...` y que puedas explicar, a partir de
la documentación, qué hace cada llamada que utilizaste.


## 7.5 Ejecutar y comparar

Desde `labs/trabajo-local/clase-10/`:

```bash
uv run python registrar_experimentos.py
```

- `uv run` usa el ambiente del proyecto sin activarlo manualmente.
- `python registrar_experimentos.py` ejecuta el archivo completo.

El resultado esperado contiene una línea por modelo con RMSE, tiempo y tamaño. Después:

1. abre `nyc-taxi-clase-10` en la interfaz;
2. selecciona los dos runs;
3. usa **Compare**;
4. revisa parameters, metrics, tags y el modelo guardado.

Responde con evidencia:

- ¿qué modelo logra menor RMSE de validación?;
- ¿cuánto cambia el tiempo de entrenamiento?;
- ¿cuánto cambia el tamaño estimado?;
- ¿qué modelo elegirías y qué restricción podría cambiar esa decisión?

Si ejecutas otra vez el script, MLflow crea runs nuevos. No sobrescribe los anteriores: cada run representa una ejecución concreta.


# 8. Ocho errores comunes

1. **Registrar únicamente el run ganador.** Se pierde el contexto que explica la elección.
2. **Cambiar datos y modelo al mismo tiempo.** No se puede atribuir la diferencia a una decisión.
3. **Usar RMSE de entrenamiento como evidencia de generalización.** La comparación debe usar abril, que no participa en el ajuste.
4. **Registrar métricas sin unidad o conjunto.** `time=2` no dice segundos, minutos, entrenamiento o inferencia.
5. **Usar nombres ambiguos.** `model-final-v2` comunica menos que `random_forest` y sus parámetros.
6. **Guardar el modelo sin firma ni ejemplo de entrada.** Después resulta difícil saber qué datos acepta.
7. **Agregar la base o los artifacts a Git.** Son estado local reconstruible y pueden crecer rápidamente.
8. **Confundir Tracking con Model Registry.** Un run conserva evidencia; versionar y asignar aliases a modelos seleccionados es otra responsabilidad.


# 9. Resumen

- Un run conecta configuración, datos, métricas y artifacts bajo una identidad.
- El seguimiento puede comenzar manualmente, automatizarse con código o apoyarse en una herramienta especializada.
- Una comparación justa conserva datos, features, target y métrica.
- Las mejores prácticas incluyen código, entorno, datos, parámetros, métricas, artifacts y decisión.
- MLflow Tracking organiza experiments y runs mediante una API, almacenamiento y una interfaz web.
- Guardar un pipeline como artifact no equivale a incorporarlo a Model Registry.


# 10. Comprobación final

Sin consultar las definiciones anteriores, explica:

1. la diferencia entre parámetro, métrica y artifact;
2. por qué marzo y abril cumplen funciones distintas;
3. qué evidencia revisarías antes de elegir entre los dos modelos;
4. qué información perderías si sólo conservaras el archivo del ganador;
5. para qué sirven `--backend-store-uri`, `--artifacts-destination`, `--host` y `--port`;
6. por qué los runs de esta clase todavía no forman un Model Registry.

No agregues `mlflow.db`, `mlartifacts/` ni `labs/trabajo-local/` a Git.


# 11. Preparación para el quiz

Al inicio de la siguiente clase habrá un quiz individual sobre los temas de las clases 8, 9 y 10.

| Clase | Temas centrales |
|---|---|
| 8 — Introducción a MLOps | propósito de MLOps, relación con DevOps, ciclo de ML, prácticas y niveles de madurez |
| 9 — Tres niveles del software de ML | datos, modelo y código; entrenamiento frente a inferencia; pipelines y patrones de serving |
| 10 — Experiment tracking | experiment, run, parámetros, métricas, tags, datasets y artifacts; pregunta e hipótesis; comparación controlada; componentes locales de MLflow |

La preparación consiste en poder explicar las relaciones entre esos conceptos y aplicarlas al caso NYC Taxi.
